# 02. Preprocessing — Credit Card Fraud Detection

**Project:** Fraud Detection & AI Governance Portfolio
**Goal:** Turn the raw data into clean, leakage-free, model-ready inputs: split into train/test, scale the right columns, and save everything for the modeling notebook.

---

### What we will do (and the one rule that matters most)
1. Separate features (`X`) from the target (`y = Class`)
2. **Split into train/test FIRST** (stratified, to preserve the 0.17% fraud ratio)
3. **Scale `Amount` and `Time`** — fitting the scaler on the training set **only**
4. Save the processed arrays + the fitted scaler to disk

> **The golden rule: no data leakage.** We split *before* scaling and fit the scaler on the training set only. If we scaled using statistics from the whole dataset (including the test set), the model would secretly "see" the test data during training and our evaluation would be a lie. Preventing leakage is exactly the kind of correctness control an AI Risk Analyst documents.

## 0. Setup + load the data

**What:** Import the tools and load the raw CSV.
**Why:** We start each notebook fresh from the original `creditcard.csv` so this stage is reproducible on its own.

In [1]:
# pandas / numpy: table handling and numerical computing (same as notebook 01).
import pandas as pd
import numpy as np

# train_test_split: splits data into training and test sets.
from sklearn.model_selection import train_test_split

# StandardScaler: rescales a column to mean 0, standard deviation 1.
from sklearn.preprocessing import StandardScaler

# joblib: saves/loads Python objects (like a fitted scaler) to disk.
import joblib

# pathlib.Path: a clean way to build and create file paths.
from pathlib import Path

print("Libraries imported ✅")

Libraries imported ✅


In [2]:
# Read the original CSV into a DataFrame (table). Same file as notebook 01.
df = pd.read_csv("../data/creditcard.csv")

# Confirm the size: should be (284807, 31).
print("Loaded shape:", df.shape)

# Peek at the first rows.
df.head()

Loaded shape: (284807, 31)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


## 1. Separate features (X) and target (y)

**What:** Split the table into `X` (the inputs the model learns from) and `y` (the answer we want to predict).
**Why:** Every supervised model needs this separation. `Class` is the label, so it must NOT be part of `X` — otherwise the model would just read the answer instead of learning.

In [3]:
# X = all columns EXCEPT 'Class'. drop(columns=...) returns a copy without that column.
X = df.drop(columns=["Class"])

# y = only the 'Class' column (0 = normal, 1 = fraud). This is what we predict.
y = df["Class"]

# Confirm the shapes. X should have 30 columns (Time, V1-V28, Amount); y is a single column.
print("X shape:", X.shape)
print("y shape:", y.shape)

# Show the fraud count in the full dataset for reference.
print("\nFraud count in full data:")
print(y.value_counts())

X shape: (284807, 30)
y shape: (284807,)

Fraud count in full data:
Class
0    284315
1       492
Name: count, dtype: int64


## 2. Train/test split (stratified) — do this BEFORE scaling

**What:** Hold out 20% of the data as a test set the model never trains on.
**Why two specific choices:**
- **`stratify=y`**: with only 0.17% fraud, a random split could put almost no fraud in the test set. Stratifying forces the **same fraud ratio** in both train and test, so the test set is a fair miniature of reality.
- **`random_state=42`**: fixes the randomness so the split is identical every time we run it → reproducible results.

**Why split first?** So that when we scale next, the scaler learns its statistics from the training set only — the test set stays truly unseen (no leakage).

In [4]:
# Split X and y into training (80%) and test (20%) sets.
# test_size=0.2  -> 20% goes to test.
# stratify=y     -> keep the same fraud/normal ratio in both sets.
# random_state=42 -> reproducible split (same result every run).
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Print the sizes of each set.
print("Train set:", X_train.shape, " Test set:", X_test.shape)

# Check that the fraud ratio was preserved in BOTH sets (.mean() of 0/1 column = fraud proportion).
print("\nFraud ratio in train: %.4f%%" % (y_train.mean() * 100))
print("Fraud ratio in test:  %.4f%%" % (y_test.mean() * 100))

Train set: (227845, 30)  Test set: (56962, 30)

Fraud ratio in train: 0.1729%
Fraud ratio in test:  0.1720%


### Interpretation
- Train ≈ 227,845 rows, test ≈ 56,962 rows.
- The fraud ratio is **~0.17% in both** sets — stratification worked. The test set is now a fair stand-in for real incoming transactions.

## 3. Scale `Amount` and `Time` (fit on train only)

**What:** Rescale `Amount` and `Time` to a comparable range (mean 0, std 1).
**Why only these two?** `V1`–`V28` already come out of PCA on a similar scale, so they don't need scaling. But `Amount` (0 to ~25,000) and `Time` (0 to ~172,000 seconds) are on huge, very different scales. Many models would let those big numbers dominate unless we standardize them.

**The leakage-safe procedure:**
1. `fit` the scaler on the **training** column → it learns that column's mean and std.
2. `transform` both train and test using those **training** statistics.

We never let the scaler look at the test set's numbers. That is what keeps evaluation honest.

In [5]:
# Columns that need scaling. V1-V28 are left as-is (already PCA-scaled).
cols_to_scale = ["Amount", "Time"]

# Create the scaler tool.
scaler = StandardScaler()

# Work on copies so we don't overwrite the original split (good habit for debugging).
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# fit_transform on TRAIN: learn mean/std from train AND apply them to train, in one step.
X_train_scaled[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])

# transform on TEST: apply the SAME train-learned mean/std to test. (transform only, never fit!)
X_test_scaled[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

# Compare before vs after for Amount: after scaling, mean should be ~0 and std ~1 on the train set.
print("BEFORE scaling (train Amount):  mean=%.2f  std=%.2f" % (X_train["Amount"].mean(), X_train["Amount"].std()))
print("AFTER  scaling (train Amount):  mean=%.2f  std=%.2f" % (X_train_scaled["Amount"].mean(), X_train_scaled["Amount"].std()))

BEFORE scaling (train Amount):  mean=88.18  std=250.72
AFTER  scaling (train Amount):  mean=0.00  std=1.00


In [6]:
# Sanity check: look at the scaled columns. Amount and Time should now sit around 0 with small spread.
X_train_scaled[["Time", "Amount"]].describe()

,Time,Amount
count,2.278450e+05,2.278450e+05
mean,-1.409578e-16,3.742243e-17
std,1.000002e+00,1.000002e+00
min,-1.998073e+00,-3.516894e-01
25%,-8.561495e-01,-3.291944e-01
50%,-2.122647e-01,-2.639429e-01
75%,9.366285e-01,-4.262209e-02
max,1.640549e+00,1.021170e+02


### Interpretation
- After scaling, the training `Amount` has **mean ≈ 0, std ≈ 1** — exactly what `StandardScaler` is supposed to produce.
- The test set was scaled with the **train's** mean/std (not its own), so there is **no leakage**.
- Note: `StandardScaler` does not remove the skew/outliers we saw in EDA. If a model struggles with the heavy tail later, a `RobustScaler` or a `log` transform on `Amount` is a reasonable alternative to revisit.

## 4. Save the processed data + the fitted scaler

**What:** Write the four arrays (`X_train`, `X_test`, `y_train`, `y_test`) and the fitted `scaler` to disk.
**Why:** The next notebook (03_Modeling) can load these directly instead of repeating all of the above — and saving the **fitted scaler** means we can apply the exact same transformation to brand-new transactions in the future. Reproducibility like this is a core governance requirement.

In [7]:
# Create a folder data/processed/ if it doesn't exist yet. parents=True makes parent dirs too.
out_dir = Path("../data/processed")
out_dir.mkdir(parents=True, exist_ok=True)

# Save each split as a CSV. index=False drops the row-number column we don't need.
X_train_scaled.to_csv(out_dir / "X_train.csv", index=False)
X_test_scaled.to_csv(out_dir / "X_test.csv", index=False)
y_train.to_csv(out_dir / "y_train.csv", index=False)
y_test.to_csv(out_dir / "y_test.csv", index=False)

# Save the FITTED scaler object so we can reuse the exact same transformation later.
joblib.dump(scaler, out_dir / "scaler.joblib")

# List what we saved.
print("Saved to", out_dir.resolve(), ":")
for f in sorted(out_dir.iterdir()):
    print("  -", f.name)

Saved to /Users/yoobinkim/Olivia/400. Self-Projects/fraud-detection-ai-governance/data/processed :
  - X_test.csv
  - X_train.csv
  - scaler.joblib
  - y_test.csv
  - y_train.csv


In [8]:
# Final verification: reload one file and confirm it round-trips correctly.
check = pd.read_csv(out_dir / "X_train.csv")
print("Reloaded X_train shape:", check.shape)
print("Scaled Amount mean (should be ~0):", round(check["Amount"].mean(), 4))

Reloaded X_train shape: (227845, 30)
Scaled Amount mean (should be ~0): 0.0


## 5. Summary

| Step | What we did | Why it matters |
|------|-------------|----------------|
| Split first | 80/20 train/test, `stratify=y` | Preserves the 0.17% fraud ratio; test set stays a fair sample |
| Scale | `StandardScaler` on `Amount`, `Time` only | Stops big-scale columns from dominating; V1-V28 already scaled |
| Fit on train only | `fit` on train, `transform` on test | **No data leakage** — honest evaluation |
| Save | 4 CSVs + `scaler.joblib` | Reproducible; reuse exact transform on new data |

> **Note on imbalance:** we did **not** resample (e.g. SMOTE) here. Resampling must be applied to the **training set only**, so we defer it to the modeling notebook where we can apply it inside a cross-validation loop. Doing it before the split would leak synthetic fraud examples into the test set.

---

### Next Step — 03_Modeling
1. Load `X_train`, `X_test`, `y_train`, `y_test` from `data/processed/`
2. Baseline: Logistic Regression with `class_weight="balanced"`
3. Compare: Random Forest / XGBoost, and resampling (SMOTE) on the training set
4. Evaluate with **Recall, Precision, PR-AUC** (not accuracy) — as decided in EDA